In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
%cd /content
!rm -rf urdu-vlm-hallucination
!git clone https://github.com/ibrahimjohar/urdu-vlm-hallucination.git
%cd urdu-vlm-hallucination

/content
Cloning into 'urdu-vlm-hallucination'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 69 (delta 8), reused 19 (delta 5), pack-reused 46 (from 1)
Receiving objects: 100% (69/69), 58.25 MiB | 16.52 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/urdu-vlm-hallucination


In [6]:
import os
print(os.listdir('/content/urdu-vlm-hallucination'))

['docs', 'scripts', 'outputs', 'LICENSE', 'notebooks', 'data', '.git', 'README.md', 'experiments', 'reference papers', 'requirements.txt', 'src', '.gitignore']


In [7]:
!cp -r /content/urdu-vlm-hallucination /content/drive/MyDrive/research/

In [9]:
import torch
print(f"GPU available : {torch.cuda.is_available()}")
print(f"GPU name : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import os
print(f"\nDataset files:")
for f in os.listdir('/content/urdu-vlm-hallucination/data/processed'):
    print(f"  {f}")

GPU available : True
GPU name : Tesla T4
VRAM : 15.6 GB

Dataset files:
  urdu_hall_bench_english.csv
  .gitkeep
  urdu_hall_bench_trilingual.csv
  urdu_visual_qa_ft.json
  urdu_hall_bench_bilingual.csv


In [10]:
!pip install -q transformers accelerate bitsandbytes peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.1 MB/s eta 0:00:00


In [11]:
!pip install -q Pillow pandas requests

In [12]:
import transformers
import accelerate
import bitsandbytes
import peft
import PIL
import pandas
import torch

print(f"transformers: {transformers.__version__}")
print(f"accelerate: {accelerate.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"peft: {peft.__version__}")
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
print("\nall imports successful")

transformers: 5.0.0
accelerate: 1.13.0
bitsandbytes: 0.49.2
peft: 0.19.1
torch: 2.10.0+cu128
cuda available: True

all imports successful


In [13]:
import os
os.environ['HF_HOME'] = '/content/hf_cache'

from transformers import LlavaForConditionalGeneration, AutoProcessor
import torch

print("loading LLaVA-1.5-7B...")
model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
print("model loaded successfully")
print(f"VRAM used : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("model loaded successfully")
print(f"model device: {model.device}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

loading LLaVA-1.5-7B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

model loaded successfully
VRAM used : 13.46 GB
model loaded successfully
model device: cuda:0
VRAM used: 13.46 GB
VRAM free: 2.18 GB


In [14]:
import pandas as pd
import torch
from PIL import Image
import requests
from pathlib import Path
import json
from tqdm import tqdm

BENCH_PATH   = Path('/content/urdu-vlm-hallucination/data/processed/urdu_hall_bench_trilingual.csv')
OUTPUT_DIR   = Path('/content/urdu-vlm-hallucination/experiments/baseline')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COCO_URL     = "http://images.cocodataset.org/val2014/{}"
LANGUAGES    = ['en', 'ur', 'roman']
QUESTION_COL = {'en': 'question_en', 'ur': 'question_ur', 'roman': 'question_roman'}
BATCH_SIZE   = 1

#load benchmark
print("loading benchmark...")
df = pd.read_csv(BENCH_PATH)
print(f"loaded {len(df)} rows")

#evaluation function
def evaluate_llava(df, language, model, processor, max_samples=None):
    results = []
    question_col = QUESTION_COL[language]
    subset = df.head(max_samples) if max_samples else df

    print(f"\nevaluating language: {language} ({len(subset)} questions)...")

    for idx, row in tqdm(subset.iterrows(), total=len(subset)):
        try:
            #load image from coco url
            url = COCO_URL.format(row['file_name'])
            image = Image.open(requests.get(url, stream=True, timeout=10).raw).convert('RGB')

            #build prompt
            question = row[question_col]
            conversation = [{
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": f"{question} Answer yes or no only."}
                ]
            }]

            prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
            inputs = processor(images=image, text=prompt, return_tensors='pt').to('cuda')

            with torch.no_grad():
                output = model.generate(**inputs, max_new_tokens=5)

            full_response = processor.decode(output[0], skip_special_tokens=True)
            assistant_response = full_response.split("ASSISTANT:")[-1].strip().lower()
            predicted = 'yes' if 'yes' in assistant_response else 'no'

            results.append({
                'image_id': row['image_id'],
                'file_name': row['file_name'],
                'question': question,
                'answer': row['answer'],
                'predicted': predicted,
                'split': row['split'],
                'category': row['category'],
                'language': language,
                'raw_response': assistant_response
            })

        except Exception as e:
            print(f"  error at idx {idx}: {e}")
            results.append({
                'image_id': row['image_id'],
                'file_name': row['file_name'],
                'question': question,
                'answer': row['answer'],
                'predicted': 'error',
                'split': row['split'],
                'category': row['category'],
                'language': language,
                'raw_response': str(e)
            })

    return pd.DataFrame(results)

#test on 10 samples first
print("running sanity check on 10 samples (english)...")
test_results = evaluate_llava(df, 'en', model, processor, max_samples=10)
print(test_results[['question', 'answer', 'predicted', 'split']].to_string())

loading benchmark...
loaded 3600 rows
running sanity check on 10 samples (english)...

evaluating language: en (10 questions)...


100%|██████████| 10/10 [00:24<00:00,  2.48s/it]

                                question answer predicted    split
0        Is there a person in the image?    yes       yes   random
1           Is there a bus in the image?    yes       yes   random
2       Is there a handbag in the image?    yes       yes   random
3  Is there a baseball bat in the image?     no        no   random
4        Is there a laptop in the image?     no        no   random
5          Is there a cake in the image?     no        no   random
6        Is there a person in the image?    yes       yes  popular
7           Is there a bus in the image?    yes       yes  popular
8       Is there a handbag in the image?    yes       yes  popular
9           Is there a car in the image?     no        no  popular


In [19]:
import os
os.chdir('/content/urdu-vlm-hallucination')
!git reset --soft HEAD~1

In [21]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!cp "/content/drive/MyDrive/Colab Notebooks/baseline_evaluation.ipynb" \
    /content/urdu-vlm-hallucination/notebooks/

!git remote set-url origin https://{token}@github.com/ibrahimjohar/urdu-vlm-hallucination.git
!git add notebooks/baseline_evaluation.ipynb
!git commit -m "add baseline evaluation notebook"
!git push origin main

[main 4257097] add baseline evaluation notebook
 1 file changed, 1 insertion(+), 1 deletion(-)
 rewrite notebooks/baseline_evaluation.ipynb (76%)
Enumerating objects: 14, done.
Counting objects: 100% (14/14), done.
Delta compression using up to 2 threads
Compressing objects: 100% (12/12), done.
Writing objects: 100% (12/12), 666.59 KiB | 12.12 MiB/s, done.
Total 12 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 1 local object.
To https://github.com/ibrahimjohar/urdu-vlm-hallucination.git
   80c74a8..4257097  main -> main


In [23]:
import pandas as pd
import torch
from PIL import Image
import requests
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

BENCH_PATH = Path('/content/urdu-vlm-hallucination/data/processed/urdu_hall_bench_trilingual.csv')
OUTPUT_DIR = Path('/content/urdu-vlm-hallucination/experiments/baseline')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COCO_URL = "http://images.cocodataset.org/val2014/{}"
LANGUAGES = ['en', 'ur', 'roman']
QUESTION_COL = {'en': 'question_en', 'ur': 'question_ur', 'roman': 'question_roman'}

#load and sample benchmark
print("loading benchmark...")
df = pd.read_csv(BENCH_PATH)

#stratified sample - 200 per language, balanced across splits and answers
def get_stratified_sample(df, n_per_split=100):
    samples = []
    for split in ['random', 'popular', 'adversarial']:
        split_df = df[df['split'] == split]
        yes_df = split_df[split_df['answer'] == 'yes'].sample(n_per_split//2, random_state=42)
        no_df = split_df[split_df['answer'] == 'no'].sample(n_per_split//2, random_state=42)
        samples.append(yes_df)
        samples.append(no_df)
    return pd.concat(samples).reset_index(drop=True)

sample_df = get_stratified_sample(df, n_per_split=100)
print(f"full benchmark: {len(df)} questions")
print(f"sampled subset: {len(sample_df)} questions")
print(f"split breakdown: {sample_df['split'].value_counts().to_dict()}")
print(f"answer balance: {sample_df['answer'].value_counts().to_dict()}")

#evaluation function
def evaluate_llava(df, language, model, processor):
    results = []
    question_col = QUESTION_COL[language]

    print(f"\nevaluating language: {language} ({len(df)} questions)...")

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            url = COCO_URL.format(row['file_name'])
            image = Image.open(requests.get(url, stream=True, timeout=10).raw).convert('RGB')

            question = row[question_col]
            conversation = [{
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": f"{question} Answer yes or no only."}
                ]
            }]

            prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
            inputs = processor(images=image, text=prompt, return_tensors='pt').to('cuda')

            with torch.no_grad():
                output = model.generate(**inputs, max_new_tokens=5)

            full_response = processor.decode(output[0], skip_special_tokens=True)
            assistant_response = full_response.split("ASSISTANT:")[-1].strip().lower()
            predicted = 'yes' if 'yes' in assistant_response else 'no'

            results.append({
                'image_id': row['image_id'],
                'file_name': row['file_name'],
                'question': question,
                'answer': row['answer'],
                'predicted': predicted,
                'split': row['split'],
                'category': row['category'],
                'language': language,
                'raw_response': assistant_response
            })

        except Exception as e:
            results.append({
                'image_id': row['image_id'],
                'file_name': row['file_name'],
                'question': question,
                'answer': row['answer'],
                'predicted': 'error',
                'split': row['split'],
                'category': row['category'],
                'language': language,
                'raw_response': str(e)
            })

    return pd.DataFrame(results)

#run evaluation across all 3 languages
all_results = []
lgs_tracker = {}

for lang in LANGUAGES:
    print(f"\n\nevaluating: {lang.upper()}")

    lang_results = evaluate_llava(sample_df, lang, model, processor)
    all_results.append(lang_results)

    #save checkpoint immediately
    lang_results.to_csv(OUTPUT_DIR / f'llava_results_{lang}.csv', index=False)
    print(f"checkpoint saved: llava_results_{lang}.csv")

    #compute metrics
    valid  = lang_results[lang_results['predicted'] != 'error']
    y_true = (valid['answer'] == 'yes').astype(int)
    y_pred = (valid['predicted'] == 'yes').astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    pr = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    hr = ((y_pred == 1) & (y_true == 0)).sum() / (y_true == 0).sum()

    lgs_tracker[lang] = hr

    print(f"\nresults for {lang.upper()}:")
    print(f"  accuracy  : {acc:.4f}")
    print(f"  precision : {pr:.4f}")
    print(f"  recall    : {rec:.4f}")
    print(f"  f1        : {f1:.4f}")
    print(f"  HR        : {hr:.4f}")

#compute language gap scores
print(f"\nLANGUAGE GAP SCORES")
print(f"  LGS (Urdu vs English)  : {lgs_tracker['ur'] - lgs_tracker['en']:.4f}")
print(f"  LGS (Roman vs English) : {lgs_tracker['roman'] - lgs_tracker['en']:.4f}")

#save combined results
combined = pd.concat(all_results, ignore_index=True)
combined.to_csv(OUTPUT_DIR / 'llava_results_all.csv', index=False)
print(f"\nall results saved to experiments/baseline/llava_results_all.csv")

loading benchmark...
full benchmark: 3600 questions
sampled subset: 300 questions
split breakdown: {'random': 100, 'popular': 100, 'adversarial': 100}
answer balance: {'yes': 150, 'no': 150}


evaluating: EN

evaluating language: en (300 questions)...


100%|██████████| 300/300 [10:38<00:00,  2.13s/it]


checkpoint saved: llava_results_en.csv

results for EN:
  accuracy  : 0.8867
  precision : 0.9143
  recall    : 0.8533
  f1        : 0.8828
  HR        : 0.0800


evaluating: UR

evaluating language: ur (300 questions)...


100%|██████████| 300/300 [10:33<00:00,  2.11s/it]


checkpoint saved: llava_results_ur.csv

results for UR:
  accuracy  : 0.6133
  precision : 0.6635
  recall    : 0.4600
  f1        : 0.5433
  HR        : 0.2333


evaluating: ROMAN

evaluating language: roman (300 questions)...


100%|██████████| 300/300 [10:29<00:00,  2.10s/it]

checkpoint saved: llava_results_roman.csv

results for ROMAN:
  accuracy  : 0.7967
  precision : 0.7764
  recall    : 0.8333
  f1        : 0.8039
  HR        : 0.2400

LANGUAGE GAP SCORES
  LGS (Urdu vs English)  : 0.1533
  LGS (Roman vs English) : 0.1600

all results saved to experiments/baseline/llava_results_all.csv


In [ ]:
import os
from google.colab import userdata

os.chdir('/content/urdu-vlm-hallucination')
token = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{token}@github.com/ibrahimjohar/urdu-vlm-hallucination.git
!git add experiments/baseline/
!git commit -m "add LLaVA baseline results - LGS Urdu=0.1533 Roman=0.1600"
!git push origin main